# CPU executable、thunk 与 warm trace

对应 R01/R02/R12。逐层检查匹配源码 003 的已封存产物；不加载 pickle/executable，也不在此内核重新编译。真正 CPU 运行及 source native reader 的独立验证见 [说明](cpu-executable-and-trace.md) 与 `cpu-thunk-results.json`。本 Notebook 执行数据解析、完整性检查、NumPy 参考和反例。

In [1]:
from pathlib import Path
import sys, json, gzip
import numpy as np
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").exists())
HERE = ROOT / "research/jax-stack"
sys.path.insert(0, str(HERE))
from cpu_executable_parser import SCHEMA, inventory, schema_pool, decode_package, project
from audit_cpu_thunks import ORIGIN, RUNTIME, trace_checks
result = json.loads((HERE / "cpu-thunk-results.json").read_text())
audit_root = ROOT / "artifacts/jax-stack/cpu-thunk-audit-002/capture"
assert inventory(audit_root, "REPLAY-OFFLINE") == result["audit_manifest"]
assert inventory(RUNTIME, "RUN-CPU") == result["runtime_capture"]
assert inventory(SCHEMA, "SOURCE-ONLY") == result["schema"]
assert result["qualifiers"] == [] and result["verification_runtime"]["source_build_verified"]
pool = schema_pool()
print("Verified archive artifacts:", result["audit_manifest"]["artifact_count"], result["runtime_capture"]["artifact_count"], result["schema"]["artifact_count"])


Verified archive artifacts: 62 39 94


## 四层包装与 thunk 类型

`pickletools.genops` 只扫描 opcode 数据。IFRT metadata 的长度前缀之后是 PJRT wrapper，其中再保存 CPU compilation proto。列出的 module ID 属于各自 capture，不能跨运行沿用。

In [2]:
for case in result["cases"]:
    for label, source in [("original", ORIGIN), ("fresh", RUNTIME)]:
        package = (source / case["case"] / "executable.bin").read_bytes()
        cpu, meta, packaging, blobs = decode_package(package, pool)
        observation = project(cpu, package)
        assert observation == case["views"][label]["projection"]
        assert packaging == case["views"][label]["packaging"]
        print(case["case"], label, "IFRT version", meta.ifrt_version_number,
              "module", observation["module_id"],
              [(t["kind"], t["op_name"]) for t in observation["thunks"]])
        for t in observation["thunks"]:
            if t["kind"] == "dot":
                print("  dot", t["lhs"]["shape"], t["rhs"]["shape"], t["dimensions"], "->", t["output"]["shape"])


matmul original IFRT version 3 module 4 [('ynn-fusion', 'ynn_fusion')]
matmul fresh IFRT version 3 module 0 [('ynn-fusion', 'ynn_fusion')]
vmap_matmul original IFRT version 3 module 14 [('ynn-fusion', 'ynn_fusion')]
vmap_matmul fresh IFRT version 3 module 3 [('ynn-fusion', 'ynn_fusion')]
grad_matmul original IFRT version 3 module 24 [('ynn-fusion', 'ynn_fusion.1'), ('kernel', 'broadcast_multiply_fusion'), ('ynn-fusion', 'ynn_fusion'), ('dot', 'dot')]
  dot [4, 8] [4, 6] {'lhs_contracting_dimensions': ['0'], 'rhs_contracting_dimensions': ['0']} -> [8, 6]
grad_matmul fresh IFRT version 3 module 6 [('ynn-fusion', 'ynn_fusion.1'), ('kernel', 'broadcast_multiply_fusion'), ('ynn-fusion', 'ynn_fusion'), ('dot', 'dot')]
  dot [4, 8] [4, 6] {'lhs_contracting_dimensions': ['0'], 'rhs_contracting_dimensions': ['0']} -> [8, 6]
jit_grad_vmap_matmul original IFRT version 3 module 34 [('ynn-fusion', 'ynn_fusion.1'), ('kernel', 'broadcast_multiply_fusion'), ('kernel', 'copy_bitcast_fusion'), ('ynn-fus

jit_grad_vmap_matmul fresh IFRT version 3 module 9 [('ynn-fusion', 'ynn_fusion.1'), ('kernel', 'broadcast_multiply_fusion'), ('kernel', 'copy_bitcast_fusion'), ('ynn-fusion', 'ynn_fusion'), ('dot', 'dot.3')]
  dot [12, 8] [6, 12] {'lhs_contracting_dimensions': ['0'], 'rhs_contracting_dimensions': ['1']} -> [8, 6]


## 实际 CPU trace 的对应

每份 trace 都来自已序列化的同一个新 compiled 对象。三个调用窗口互不重叠，并且阻塞到结果就绪。完成事件没有导出的 run_id；这里的对应不能推广成并发同名事件的通用拼接方法，producer dur 也不是通用异步完整时长。

In [3]:
starts = ends = 0
for case in result["cases"]:
    path = next((RUNTIME / case["case"] / "trace").rglob("perfetto_trace.json.gz"))
    with gzip.open(path, "rt") as stream:
        events = json.load(stream)["traceEvents"]
    observed = trace_checks(events, case["views"]["fresh"]["projection"], case["case"])
    assert observed == case["runtime_trace"]
    starts += observed["producer_events"]; ends += observed["completion_events"]
    print(case["case"], observed["invocations"])
assert starts == ends == 33
print("33 producer and 33 completion events; CPU host-thread observations only.")


matmul [{'iteration': 0, 'run_id': '-113118168', 'producer_count': 1, 'completion_count': 1, 'worker_thread_differs_from_python': False}, {'iteration': 1, 'run_id': '-113118167', 'producer_count': 1, 'completion_count': 1, 'worker_thread_differs_from_python': False}, {'iteration': 2, 'run_id': '-113118166', 'producer_count': 1, 'completion_count': 1, 'worker_thread_differs_from_python': False}]
vmap_matmul [{'iteration': 0, 'run_id': '-1085204305', 'producer_count': 1, 'completion_count': 1, 'worker_thread_differs_from_python': True}, {'iteration': 1, 'run_id': '-1085204304', 'producer_count': 1, 'completion_count': 1, 'worker_thread_differs_from_python': True}, {'iteration': 2, 'run_id': '-1085204303', 'producer_count': 1, 'completion_count': 1, 'worker_thread_differs_from_python': True}]
grad_matmul [{'iteration': 0, 'run_id': '1467431885', 'producer_count': 4, 'completion_count': 4, 'worker_thread_differs_from_python': True}, {'iteration': 1, 'run_id': '1467431886', 'producer_count'

## 数值结果与内存表示的边界

独立 float64 参考重新检查 12 次已保存运行。slice 的 allocation/offset/size 是编译产物坐标；本节没有测运行时指针、实际流量或物理峰值。

In [4]:
with np.load(RUNTIME / "inputs.npz", allow_pickle=False) as inputs:
    a, w, x = [inputs[k].astype(np.float64) for k in ("a", "w", "x")]
y, z = a@w, x@w
references = {"matmul": (y,), "vmap_matmul": (z,), "grad_matmul": (2*y@w.T, 2*a.T@y),
              "jit_grad_vmap_matmul": (2*z@w.T, 2*np.einsum("bmk,bmn->kn", x, z))}
for name, refs in references.items():
    with np.load(RUNTIME / name / "outputs.npz", allow_pickle=False) as outputs:
        for iteration in range(3):
            for i, reference in enumerate(refs):
                np.testing.assert_allclose(outputs[f"iteration_{iteration}_output_{i}"], reference, rtol=2e-5, atol=2e-5)
    print(name, "3 saved CPU invocations passed")


matmul 3 saved CPU invocations passed
vmap_matmul 3 saved CPU invocations passed
grad_matmul 3 saved CPU invocations passed
jit_grad_vmap_matmul 3 saved CPU invocations passed


## 延迟加载与反例

首轮环境相等断言失败，第二轮允许新增已映射库，但前后清单各自绑定同一源码 wheel，原有字节与其余环境必须相等。下面重查该差异及 19 个数据/结构/事件反例；完整 native wheel 绑定由独立固定镜像进程验证。

In [5]:
from cpu_thunk_probe import check_environment_transition
from verify_cpu_thunks import selftest
before = json.loads((RUNTIME / "environment-before.json").read_text())
after = json.loads((RUNTIME / "environment.json").read_text())
assert check_environment_transition(before, after) == ["mlir/_mlir_libs/_mlirHlo.so"]
negative_tests = selftest(audit_root)
assert negative_tests == result["negative_tests"] and len(negative_tests) == 19
print("New source-bound mapped library:", result["newly_loaded_native_libraries"])
print("Rejected counterexamples:", negative_tests)


New source-bound mapped library: ['mlir/_mlir_libs/_mlirHlo.so']
Rejected counterexamples: ['hash', 'missing', 'level', 'wrong-persistent-id', 'ambiguous-byte-payload', 'trailing-pickle-bytes', 'invalid-metadata-varint', 'unknown-protobuf-field', 'kind-oneof-disagree', 'dot-dimensions-disagree', 'wrong-ynn-instruction', 'slice-outside-allocation', 'wrong-kernel-symbol', 'duplicate-object-bytes', 'missing-dot-events', 'merged-run-identities', 'completion-outside-window', 'existing-native-payload-changed', 'source-changed-during-run']
